### 07 — Multimodal Structured Report Generation

What this notebook does:

- build structured reference reports from the BrEaST annotations
- load GPT-2 and prepare its tokenizer
- create the report-generation dataset
- load the trained U-Net, EfficientNet, ClinicalBERT, and multimodal fusion components
- generate the same `1536`-dimensional multimodal features used by the classifier
- condition GPT-2 on the multimodal representation
- add a dedicated `<END_REPORT>` stopping token
- train the multimodal projection and selected GPT-2 layers
- save the report generator with the lowest validation loss.

The report generator produces the descriptive findings and BI-RADS assessment. The final benign/malignant classification is handled separately by the dedicated classifier in the integrated system.


### 1. Imports and Project Setup


In [ ]:
from pathlib import Path
import sys
import json
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    GPT2Tokenizer,
    GPT2LMHeadModel
)

"""
If using Google Colab, uncomment the following lines to mount your Google Drive and set the project root directory. 
Otherwise, the project root is set to the parent directory of the current script.
"""
# from google.colab import drive

# drive.mount("/content/drive")

# PROJECT_ROOT = Path(
#     "/content/drive/MyDrive/Breast_Ultrasound_FYP/breast-ultrasound-fyp"
# )

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.unet import UNet
from src.image_encoder import EfficientNetEncoder
from src.clinical_encoder import ClinicalBERTEncoder
from src.fusion import CrossAttentionFusion
from src.report_generator import MultimodalReportGenerator


### 2. Define Paths, Load the Dataset Splits, and Set the Device

The same train, validation, and test splits used by the previous notebooks are reused here.

CUDA is used when a compatible GPU is available.


In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

IMAGE_DIR = RAW_DIR / "images"
MODEL_DIR = PROJECT_ROOT / "models"

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "val.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Project root:", PROJECT_ROOT)
print("Images:", IMAGE_DIR.exists())
print("Models:", MODEL_DIR.exists())
print("Splits:", len(train_df), len(val_df), len(test_df))
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


### 3. Build the Structured Reference Reports

The BrEaST dataset contains structured annotations rather than unrestricted free-text radiology reports.

These annotations are converted into a standard report template containing:

- tissue composition
- shape
- margin
- echogenicity
- posterior features
- halo
- calcifications
- skin thickening
- BI-RADS assessment.

<END_REPORT> is added to mark the end of each target report.


In [ ]:
def clean_value(value):
    if pd.isna(value):
        return "not available"

    value = str(value).strip()

    if value == "":
        return "not available"

    return value.replace("&", ", ")


def normalize_yes_no(value):
    value = clean_value(value).lower()

    if value == "yes":
        return "present"

    if value == "no":
        return "absent"

    return value


def normalize_posterior_features(value):
    value = clean_value(value)

    if value.lower() == "no":
        return "none"

    return value


def create_reference_report(row):
    return (
        "Findings: "
        f"Tissue composition: {clean_value(row['Tissue_composition'])}. "
        f"Shape: {clean_value(row['Shape'])}. "
        f"Margin: {clean_value(row['Margin'])}. "
        f"Echogenicity: {clean_value(row['Echogenicity'])}. "
        f"Posterior features: {normalize_posterior_features(row['Posterior_features'])}. "
        f"Halo: {normalize_yes_no(row['Halo'])}. "
        f"Calcifications: {normalize_yes_no(row['Calcifications'])}. "
        f"Skin thickening: {normalize_yes_no(row['Skin_thickening'])}. "
        f"Assessment: BI-RADS {clean_value(row['BIRADS'])}. "
        "<END_REPORT>"
    )


for df in (train_df, val_df, test_df):
    df["reference_report"] = df.apply(
        create_reference_report,
        axis=1
    )

print(train_df.iloc[30]["reference_report"])


### 4. Load GPT-2 and Configure Padding


In [ ]:
GPT2_MODEL_NAME = "gpt2"

gpt2_tokenizer = GPT2Tokenizer.from_pretrained(
    GPT2_MODEL_NAME
)

gpt2_model = GPT2LMHeadModel.from_pretrained(
    GPT2_MODEL_NAME
).to(device)

gpt2_tokenizer.pad_token = (
    gpt2_tokenizer.eos_token
)

gpt2_model.config.pad_token_id = (
    gpt2_tokenizer.eos_token_id
)


### 5. Test GPT-2 Tokenisation

A sample reference report is tokenised to confirm that the report text can be represented correctly.

Reports are limited to a maximum length of 128 tokens.


In [ ]:
sample_report = train_df.iloc[0]["reference_report"]

encoded_report = gpt2_tokenizer(
    sample_report,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

print("Input shape:", encoded_report["input_ids"].shape)
print(
    gpt2_tokenizer.decode(
        encoded_report["input_ids"][0]
    )
)


### 6. Test the Report Generator with Dummy Multimodal Features

The multimodal classifier produces a 1536-dimensional feature vector, while GPT-2 uses a 768-dimensional embedding space.

MultimodalReportGenerator contains a projection layer that converts the multimodal vector into the GPT-2 embedding dimension.


In [ ]:
reports = train_df["reference_report"].iloc[:4].tolist()

report_batch = gpt2_tokenizer(
    reports,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

report_batch = {
    key: value.to(device)
    for key, value in report_batch.items()
}

labels = report_batch["input_ids"].clone()
labels[report_batch["attention_mask"] == 0] = -100

test_report_generator = MultimodalReportGenerator(
    multimodal_dim=1536
).to(device)

dummy_multimodal_features = torch.randn(
    4,
    1536,
    device=device
)

outputs = test_report_generator(
    multimodal_features=dummy_multimodal_features,
    input_ids=report_batch["input_ids"],
    attention_mask=report_batch["attention_mask"],
    labels=labels
)

print("Loss:", outputs.loss.item())
print("Logits:", outputs.logits.shape)


### 7. Create the Report-Generation Dataset

Each sample contains:

- the grayscale ultrasound image
- ClinicalBERT token IDs and attention mask
- optional token-type IDs
- GPT-2 report token IDs
- GPT-2 attention mask
- report labels.


In [ ]:
class BreastReportDataset(Dataset):
    def __init__(
        self,
        dataframe,
        image_dir,
        clinical_tokenizer,
        report_tokenizer,
        image_size=224,
        clinical_max_length=64,
        report_max_length=128
    ):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.clinical_tokenizer = clinical_tokenizer
        self.report_tokenizer = report_tokenizer
        self.image_size = image_size
        self.clinical_max_length = clinical_max_length
        self.report_max_length = report_max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(
            self.image_dir / row["Image_filename"]
        ).convert("L")

        image = image.resize(
            (self.image_size, self.image_size),
            Image.BILINEAR
        )

        image = torch.tensor(
            np.array(image, dtype=np.float32) / 255.0,
            dtype=torch.float32
        ).unsqueeze(0)

        clinical_encoded = self.clinical_tokenizer(
            row["clinical_text"],
            padding="max_length",
            truncation=True,
            max_length=self.clinical_max_length,
            return_tensors="pt"
        )

        report_encoded = self.report_tokenizer(
            row["reference_report"],
            padding="max_length",
            truncation=True,
            max_length=self.report_max_length,
            return_tensors="pt"
        )

        report_input_ids = (
            report_encoded["input_ids"].squeeze(0)
        )

        report_attention_mask = (
            report_encoded["attention_mask"].squeeze(0)
        )

        report_labels = report_input_ids.clone()
        report_labels[report_attention_mask == 0] = -100

        sample = {
            "image": image,
            "input_ids": clinical_encoded["input_ids"].squeeze(0),
            "attention_mask": clinical_encoded["attention_mask"].squeeze(0),
            "report_input_ids": report_input_ids,
            "report_attention_mask": report_attention_mask,
            "report_labels": report_labels
        }

        token_type_ids = clinical_encoded.get(
            "token_type_ids"
        )

        if token_type_ids is not None:
            sample["token_type_ids"] = token_type_ids.squeeze(0)

        return sample


### 8. Create the Report Datasets and Data Loaders

The original report-training configuration is preserved:

- report batch size: `1`
- training loader shuffled
- validation and test loaders not shuffled.


In [ ]:
CLINICAL_MODEL_NAME = (
    "emilyalsentzer/Bio_ClinicalBERT"
)

clinical_tokenizer = AutoTokenizer.from_pretrained(
    CLINICAL_MODEL_NAME
)

train_report_dataset = BreastReportDataset(
    train_df,
    IMAGE_DIR,
    clinical_tokenizer,
    gpt2_tokenizer
)

val_report_dataset = BreastReportDataset(
    val_df,
    IMAGE_DIR,
    clinical_tokenizer,
    gpt2_tokenizer
)

test_report_dataset = BreastReportDataset(
    test_df,
    IMAGE_DIR,
    clinical_tokenizer,
    gpt2_tokenizer
)

REPORT_BATCH_SIZE = 1

train_report_loader = DataLoader(
    train_report_dataset,
    batch_size=REPORT_BATCH_SIZE,
    shuffle=True
)

val_report_loader = DataLoader(
    val_report_dataset,
    batch_size=REPORT_BATCH_SIZE,
    shuffle=False
)

test_report_loader = DataLoader(
    test_report_dataset,
    batch_size=REPORT_BATCH_SIZE,
    shuffle=False
)

batch = next(iter(train_report_loader))

print("Images:", batch["image"].shape)
print("Clinical IDs:", batch["input_ids"].shape)
print("Report IDs:", batch["report_input_ids"].shape)


### 9. Load the Frozen Multimodal Feature Extractors

The report generator uses the trained upstream pipeline:

- U-Net for automatic lesion segmentation
- EfficientNet-B0 for lesion-guided image features
- ClinicalBERT for clinical-text features
- bidirectional cross-attention for multimodal fusion.

The EfficientNet and fusion weights are loaded from `best_multimodal_classifier.pth`.


In [ ]:
# U-Net
segmentation_model = UNet().to(device)
segmentation_model.load_state_dict(
    torch.load(
        MODEL_DIR / "best_unet.pth",
        map_location=device
    )
)
segmentation_model.eval()

for parameter in segmentation_model.parameters():
    parameter.requires_grad = False


# Segmentation threshold
with open(
    MODEL_DIR / "segmentation_config.json",
    "r"
) as f:
    segmentation_config = json.load(f)

BEST_THRESHOLD = segmentation_config[
    "best_threshold"
]

print("Loaded BEST_THRESHOLD:", BEST_THRESHOLD)


# Image and clinical encoders
image_encoder = EfficientNetEncoder(
    output_dim=768
).to(device)

clinical_encoder = ClinicalBERTEncoder(
    model_name=CLINICAL_MODEL_NAME
).to(device)

fusion_model = CrossAttentionFusion(
    embed_dim=768,
    num_heads=8,
    dropout=0.1
).to(device)


# Load trained multimodal states
checkpoint = torch.load(
    MODEL_DIR / "best_multimodal_classifier.pth",
    map_location=device
)

image_encoder.load_state_dict(
    checkpoint["image_encoder"]
)

fusion_model.load_state_dict(
    checkpoint["fusion_model"]
)


# Freeze upstream multimodal components
for model in (
    image_encoder,
    clinical_encoder,
    fusion_model
):
    for parameter in model.parameters():
        parameter.requires_grad = False

    model.eval()


### 10. Define Multimodal Feature Extraction

The same upstream feature pipeline is used as in the classification notebook.

The predicted U-Net mask guides EfficientNet, while ClinicalBERT processes the clinical text. The fused visual and clinical vectors are concatenated into a 1536-dimensional representation.


In [ ]:
def masked_mean_pool(tokens, attention_mask):
    mask = attention_mask.unsqueeze(-1).float()

    summed = (tokens * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)

    return summed / counts


def extract_multimodal_features(
    images,
    input_ids,
    attention_mask,
    token_type_ids=None
):
    with torch.no_grad():
        segmentation_logits = segmentation_model(
            images
        )

        predicted_masks = (
            torch.sigmoid(segmentation_logits)
            >= BEST_THRESHOLD
        ).float()

    rgb_images = images.repeat(1, 3, 1, 1)

    mean = torch.tensor(
        [0.485, 0.456, 0.406],
        device=device
    ).view(1, 3, 1, 1)

    std = torch.tensor(
        [0.229, 0.224, 0.225],
        device=device
    ).view(1, 3, 1, 1)

    rgb_images = (rgb_images - mean) / std

    with torch.no_grad():
        visual_tokens = image_encoder(
            rgb_images,
            predicted_masks
        )

        clinical_tokens = clinical_encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        visual_fused, clinical_fused = (
            fusion_model(
                visual_tokens,
                clinical_tokens,
                clinical_padding_mask=(
                    attention_mask == 0
                )
            )
        )

    visual_vector = visual_fused.mean(dim=1)

    clinical_vector = masked_mean_pool(
        clinical_fused,
        attention_mask
    )

    return torch.cat(
        [visual_vector, clinical_vector],
        dim=1
    )


### 11. Test the Multimodal Feature Pipeline

A training batch is passed through the frozen upstream components


In [ ]:
batch = next(iter(train_report_loader))

images = batch["image"].to(device)
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)

token_type_ids = batch.get("token_type_ids")

if token_type_ids is not None:
    token_type_ids = token_type_ids.to(device)

multimodal_features = extract_multimodal_features(
    images,
    input_ids,
    attention_mask,
    token_type_ids
)

print(multimodal_features.shape)


### 12. Configure the GPT-2 Report Generator

A new MultimodalReportGenerator is created using:

- GPT-2
- 1536 multimodal input dimensions
- the <END_REPORT> special token.


In [ ]:
report_generator = MultimodalReportGenerator(
    model_name="gpt2",
    multimodal_dim=1536
).to(device)

report_generator.gpt2.config.pad_token_id = (
    gpt2_tokenizer.eos_token_id
)

special_tokens = {
    "additional_special_tokens": [
        "<END_REPORT>"
    ]
}

gpt2_tokenizer.add_special_tokens(
    special_tokens
)

report_generator.gpt2.resize_token_embeddings(
    len(gpt2_tokenizer)
)

END_REPORT_ID = (
    gpt2_tokenizer.convert_tokens_to_ids(
        "<END_REPORT>"
    )
)

print("END_REPORT_ID:", END_REPORT_ID)


### 13. Select the Trainable Report-Generation Parameters

The original fine-tuning configuration is preserved.

The trainable components are:

- multimodal projection layer
- final two GPT-2 transformer blocks
- final GPT-2 layer normalisation
- language-model head.

The remaining upstream multimodal components stay frozen.


In [ ]:
for parameter in (
    report_generator
    .multimodal_projection
    .parameters()
):
    parameter.requires_grad = True

for block in (
    report_generator
    .gpt2
    .transformer
    .h[-2:]
):
    for parameter in block.parameters():
        parameter.requires_grad = True

for parameter in (
    report_generator
    .gpt2
    .transformer
    .ln_f
    .parameters()
):
    parameter.requires_grad = True

for parameter in (
    report_generator
    .gpt2
    .lm_head
    .parameters()
):
    parameter.requires_grad = True

trainable = sum(
    parameter.numel()
    for parameter in report_generator.parameters()
    if parameter.requires_grad
)

total = sum(
    parameter.numel()
    for parameter in report_generator.parameters()
)

print("Trainable:", trainable)
print("Total:", total)
print("Trainable %:", 100 * trainable / total)


### 14. Configure Report-Generator Training

The original training settings are kept unchanged:

- AdamW optimiser
- learning rate: `5e-5`
- weight decay: `1e-4`
- epochs: `10`
- checkpoint selection: lowest validation loss.


In [ ]:
optimizer = torch.optim.AdamW(
    filter(
        lambda parameter: parameter.requires_grad,
        report_generator.parameters()
    ),
    lr=5e-5,
    weight_decay=1e-4
)

REPORT_EPOCHS = 10

train_report_losses = []
val_report_losses = []

best_val_loss = float("inf")

gc.collect()
torch.cuda.empty_cache()


### 15. Helper for Moving Report Batches to the Device

This keeps the training and validation loops shorter while preserving the same tensors used by the original notebook.


In [ ]:
def prepare_report_batch(batch):
    token_type_ids = batch.get(
        "token_type_ids"
    )

    if token_type_ids is not None:
        token_type_ids = token_type_ids.to(
            device
        )

    return (
        batch["image"].to(device),
        batch["input_ids"].to(device),
        batch["attention_mask"].to(device),
        token_type_ids,
        batch["report_input_ids"].to(device),
        batch["report_attention_mask"].to(device),
        batch["report_labels"].to(device)
    )


### 16. Train the Report Generator

For every batch:

1. the frozen upstream pipeline produces the multimodal feature vector;
2. GPT-2 receives the multimodal representation and reference report tokens;
3. the language-model loss is calculated;
4. only the selected report-generator parameters are updated.

The model with the lowest validation loss is saved as best_report_generator.pth.


In [ ]:
for epoch in range(REPORT_EPOCHS):
    # -------------------------
    # Training
    # -------------------------
    report_generator.train()
    running_train_loss = 0.0

    for batch in train_report_loader:
        (
            images,
            input_ids,
            attention_mask,
            token_type_ids,
            report_input_ids,
            report_attention_mask,
            report_labels
        ) = prepare_report_batch(batch)

        with torch.no_grad():
            multimodal_features = (
                extract_multimodal_features(
                    images,
                    input_ids,
                    attention_mask,
                    token_type_ids
                )
            )

        optimizer.zero_grad()

        outputs = report_generator(
            multimodal_features=multimodal_features,
            input_ids=report_input_ids,
            attention_mask=report_attention_mask,
            labels=report_labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        running_train_loss += loss.item()

    train_loss = (
        running_train_loss
        / len(train_report_loader)
    )

    # -------------------------
    # Validation
    # -------------------------
    report_generator.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for batch in val_report_loader:
            (
                images,
                input_ids,
                attention_mask,
                token_type_ids,
                report_input_ids,
                report_attention_mask,
                report_labels
            ) = prepare_report_batch(batch)

            multimodal_features = (
                extract_multimodal_features(
                    images,
                    input_ids,
                    attention_mask,
                    token_type_ids
                )
            )

            outputs = report_generator(
                multimodal_features=multimodal_features,
                input_ids=report_input_ids,
                attention_mask=report_attention_mask,
                labels=report_labels
            )

            running_val_loss += (
                outputs.loss.item()
            )

    val_loss = (
        running_val_loss
        / len(val_report_loader)
    )

    train_report_losses.append(
        train_loss
    )

    val_report_losses.append(
        val_loss
    )

    print(
        f"Epoch {epoch + 1}/{REPORT_EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            report_generator.state_dict(),
            MODEL_DIR / "best_report_generator.pth"
        )

        print(
            "Best report generator saved."
        )


### 17. Review the Report-Generation Loss

The training and validation loss curves are plotted to show convergence and possible overfitting.


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    train_report_losses,
    label="Training Loss"
)

plt.plot(
    val_report_losses,
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()
